<a href="https://colab.research.google.com/github/ZeninKris/zmm-movilidad-predictiva/blob/main/notebooks/04_EDA_OCISEVI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELDA 1 — Carga y reconocimiento inicial de los 3 RATIV
# Objetivo: entender qué tenemos antes de tocar cualquier dato
# ============================================================

import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

RAW = '/content/drive/MyDrive/Proyecto_ZMM/data_raw/'

# Cargar los 3 archivos
df_23 = pd.read_csv(RAW + 'rativ_abierto_2023.csv', encoding='latin1', low_memory=False)
df_24 = pd.read_csv(RAW + 'rativ_abierto_2024.csv', encoding='latin1', low_memory=False)
df_25 = pd.read_csv(RAW + 'rativ_abierto_2025.csv', encoding='latin1', low_memory=False)

# Vista rápida de cada año
for año, df in [('2023', df_23), ('2024', df_24), ('2025', df_25)]:
    print(f"\n{'='*50}")
    print(f"  RATIV {año} — {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"{'='*50}")
    print(df.dtypes)

Mounted at /content/drive

  RATIV 2023 — 73,107 filas × 121 columnas
MUNICIPIO                          int64
ENCUESTA                           int64
DÃ­a                               int64
Mes                                int64
AÃ±o                               int64
                                   ...  
Fallecido 3                        int64
Dictamen MÃ©dico.2                 int64
Tipo de Ebriedad.2                 int64
Conductor a disposiciÃ³n de .2     int64
ManifestaciÃ³n de los hechos.2    object
Length: 121, dtype: object

  RATIV 2024 — 67,956 filas × 128 columnas
MUNICIPIO                          int64
ENCUESTA                           int64
DÃ­a                               int64
Mes                                int64
AÃ±o                               int64
                                   ...  
Dictamen MÃ©dico.2                 int64
Tipo de Ebriedad.2                 int64
Conductor a disposiciÃ³n de .2     int64
ManifestaciÃ³n de los hechos.2    objec

In [2]:
# ============================================================
# CELDA 2 — Diagnóstico: encoding, coordenadas y columnas
# ============================================================

# --- 2.1 Corregir encoding de nombres de columna ---
def fix_encoding(df):
    df.columns = [c.encode('latin1').decode('utf-8', errors='replace') for c in df.columns]
    return df

df_23 = fix_encoding(df_23)
df_24 = fix_encoding(df_24)
df_25 = fix_encoding(df_25)

print("✅ Encoding de columnas corregido")
print("\nPrimeras 10 columnas de 2023:")
print(df_23.columns[:10].tolist())

# --- 2.2 Buscar columna de coordenadas ---
print("\n🔍 ¿Existe columna 'Referencia'?")
for año, df in [('2023', df_23), ('2024', df_24), ('2025', df_25)]:
    ref_cols = [c for c in df.columns if 'efer' in c.lower() or 'lat' in c.lower() or 'lon' in c.lower() or 'coord' in c.lower()]
    print(f"  {año}: {ref_cols if ref_cols else '❌ No encontrada'}")

# --- 2.3 Columnas en común vs columnas únicas ---
cols_23 = set(df_23.columns)
cols_24 = set(df_24.columns)
cols_25 = set(df_25.columns)

comunes = cols_23 & cols_24 & cols_25
solo_24 = cols_24 - cols_23 - cols_25
solo_25 = cols_25 - cols_23 - cols_24

print(f"\n📊 Columnas en común (los 3 años): {len(comunes)}")
print(f"   Solo en 2024: {solo_24}")
print(f"   Solo en 2025: {solo_25}")

# --- 2.4 Columnas basura (Unnamed) ---
print("\n🗑️ Columnas basura encontradas:")
for año, df in [('2023', df_23), ('2024', df_24), ('2025', df_25)]:
    basura = [c for c in df.columns if 'Unnamed' in c]
    print(f"  {año}: {basura}")

✅ Encoding de columnas corregido

Primeras 10 columnas de 2023:
['MUNICIPIO', 'ENCUESTA', 'Día', 'Mes', 'Año', 'Día de la semana', 'Hora de reporte', 'Hora de asignación', 'Calle', 'Colonia']

🔍 ¿Existe columna 'Referencia'?
  2023: ['Colonia', 'Referencia']
  2024: ['Colonia', 'Referencia']
  2025: ['Colonia', 'Referencia']

📊 Columnas en común (los 3 años): 119
   Solo en 2024: {'Tipo de vialidad', 'Unnamed: 249'}
   Solo en 2025: {'Unnamed: 229', 'No de  Infraccion', 'Unnamed: 227'}

🗑️ Columnas basura encontradas:
  2023: []
  2024: ['Unnamed: 249']
  2025: ['Unnamed: 227', 'Unnamed: 229']


In [3]:
# ============================================================
# CELDA 3 — ¿Cómo vienen las coordenadas? + limpieza inicial
# ============================================================

# --- 3.1 Ver ejemplos reales de la columna Referencia ---
print("🗺️ Ejemplos de la columna 'Referencia':\n")
for año, df in [('2023', df_23), ('2024', df_24), ('2025', df_25)]:
    muestra = df['Referencia'].dropna().head(5).tolist()
    print(f"  {año}:")
    for v in muestra:
        print(f"    → {v}")
    print()

# --- 3.2 ¿Cuántos siniestros NO tienen coordenadas? ---
print("❓ Nulos en 'Referencia':")
for año, df in [('2023', df_23), ('2024', df_24), ('2025', df_25)]:
    nulos = df['Referencia'].isna().sum()
    pct = nulos / len(df) * 100
    print(f"  {año}: {nulos:,} sin coordenadas ({pct:.1f}%)")

# --- 3.3 Eliminar columnas basura Unnamed ---
basura = ['Unnamed: 249', 'Unnamed: 227', 'Unnamed: 229']
df_24 = df_24.drop(columns=[c for c in basura if c in df_24.columns])
df_25 = df_25.drop(columns=[c for c in basura if c in df_25.columns])

print("\n✅ Columnas basura eliminadas")
print(f"   2024 ahora: {df_24.shape[1]} columnas")
print(f"   2025 ahora: {df_25.shape[1]} columnas")

🗺️ Ejemplos de la columna 'Referencia':

  2023:
    → 99
    → 99
    → 99
    → 99
    → 99

  2024:
    → 25.653085, -100.397545
    → 25.671047, -100.397998
    → 25.686129, -100.416994
    → 25.663848, -100.381837
    → 25.650216, -100.358671

  2025:
    → 25.671176, -100.396215
    → 25.683424, -100.417420
    → 25.647660, -100.332841
    → 25.661059, -100.410031
    → 25.675487, -100.417683

❓ Nulos en 'Referencia':
  2023: 1 sin coordenadas (0.0%)
  2024: 0 sin coordenadas (0.0%)
  2025: 0 sin coordenadas (0.0%)

✅ Columnas basura eliminadas
   2024 ahora: 127 columnas
   2025 ahora: 127 columnas
